In [1]:
import glob
import re
from nilearn.glm.second_level import SecondLevelModel
import pandas as pd
import nibabel as nf
import numpy as np
from pathlib import Path

from survey_medley_code.config_loader import load_config
from subject_slope_contrasts import get_sub_event_file, define_questionnaires, get_behavior_bold_data, apply_inclusion_criteria, build_design_matrix_contrasts, run_within_subject_questionnaire_glm

In [2]:
def apply_inclusion_criteria(df, questionnaires, min_obs=5, min_unique=3, min_obs_per_level_when_binary=None):
    omissions_by_questionnaire = {}
    for questionnaire in questionnaires:
        omissions_by_questionnaire[questionnaire] = False
        print("processing", questionnaire)
        omit = False
        filtered_df = df[df['questionnaire_name'] == questionnaire]
        num_obs = filtered_df.shape[0]
        print("observations:", num_obs)
        if num_obs < min_obs:
            omit = True
            print(num_obs, "observations <", min_obs, "minimum")
        num_unique = filtered_df['behavior'].nunique()
        if num_unique < min_unique:
            omit = True
            print(num_unique, "unique obs <", min_unique, "minimum")
        if min_obs_per_level_when_binary != None and num_unique == 2:
            num_obs_per_level_series = filtered_df['behavior'].value_counts()
            lowest_obs_per_level = num_obs_per_level_series.min()
            if lowest_obs_per_level < min_obs_per_level_when_binary:
                omit = True
                print(lowest_obs_per_level, "obs per level when binary <", min_obs_per_level_when_binary, "minimum")
        if omit:
            df = df[df['questionnaire_name'] != questionnaire]
            print("omitting", questionnaire)
            omissions_by_questionnaire[questionnaire] = True
    new_df = df.reset_index(drop=True)
    return new_df, omissions_by_questionnaire

In [3]:
cfg = load_config()
question_output_path = (
cfg.output_root / 'within_subject_question_estimates/within_subject_results'
)
root = '/oak/stanford/groups/russpold/data/uh2/aim1'

activation_maps = glob.glob(f'{question_output_path}/*/*.nii.gz')
sub_ids = sorted(set([re.search('_sub_(.*).nii.gz', val).group(1) for val in activation_maps]))

# Load events files
events_files = sorted(glob.glob(f'{root}/BIDS/sub-s*/ses-[0-9]/func/*surveyMedley*modified*.tsv'))

questionnaires = define_questionnaires()

omissions_by_questionnaire = {"grit": 0, "brief": 0, "future_time": 0, "upps": 0, "impulsive_venture": 0}
# for sub_id in sub_ids:
#     print("testing on subject", sub_id)
#     # run_within_subject_questionnaire_glm(sub_id, events_files, questionnaires, question_output_path, activation_maps)
#     df = get_behavior_bold_data(sub_id, events_files, questionnaires, question_output_path, activation_maps)
#     new_df, subject_omissions = apply_inclusion_criteria(df, questionnaires, min_obs=5, min_unique=3, min_obs_per_level_when_binary=None)
#     for questionnaire in subject_omissions:
#         if subject_omissions[questionnaire]:
#             omissions_by_questionnaire[questionnaire] += 1
# print(omissions_by_questionnaire)

df = get_behavior_bold_data('495', events_files, questionnaires, question_output_path, activation_maps)
new_df, subject_omissions = apply_inclusion_criteria(df, questionnaires, min_obs=5, min_unique=3, min_obs_per_level_when_binary=None)
print(new_df)

# design_matrix, sub_bold_files, contrast_names = build_design_matrix_contrasts(new_df)
# print(sub_bold_files)
# print(contrast_names)
# design_matrix

processing grit
observations: 6
2 unique obs < 3 minimum
omitting grit
processing brief
observations: 9
processing future_time
observations: 8
processing upps
observations: 4
4 observations < 5 minimum
2 unique obs < 3 minimum
omitting upps
processing impulsive_venture
observations: 2
2 observations < 5 minimum
1 unique obs < 3 minimum
omitting impulsive_venture
   questionnaire_name question_id  behavior  chr_count  \
0               brief         Q09      0.25         34   
1               brief         Q12      1.00         27   
2               brief         Q13      0.75         57   
3               brief         Q14      0.50         36   
4               brief         Q15      0.50         34   
5               brief         Q16      0.50         50   
6               brief         Q17      0.50         58   
7               brief         Q19      0.75         53   
8               brief         Q20      1.00         79   
9         future_time         Q23      0.75         54 